In [0]:
%pip install scikit-learn mlflow
dbutils.library.restartPython()

In [0]:
   %pip install scikit-learn mlflow xgboost lightgbm imbalanced-learn
   dbutils.library.restartPython()

In [0]:
# load the data and label spikes

df = spark.sql("""
    select
        region_code,
        interval_start_at,
        cast(price_aud_per_mwh as double) as price_aud_per_mwh,
        cast(demand_mw as double)         as demand_mw,
        hour(interval_start_at)           as hour_of_day,
        weekday(date_day) + 1             as day_of_week,
        case when weekday(date_day) >= 5 then 1 else 0 end as is_weekend
    from workspace.gold.fact_dispatch_price
""").toPandas()

df["is_spike"] = (df["price_aud_per_mwh"] > 300).astype(int)

print(df["is_spike"].value_counts())
print(f"Spike rate: {df['is_spike'].mean():.2%}")

In [0]:
# build features and split by time

import pandas as pd

# Put each state's intervals in time order. The data has no gaps, so 6 rows back = 30 minutes back.
df = df.sort_values(["region_code", "interval_start_at"]).reset_index(drop=True)
df["price_aud_per_mwh"] = df["price_aud_per_mwh"].astype(float)
df["demand_mw"] = df["demand_mw"].astype(float)
df["is_weekend"] = df["is_weekend"].astype(int)

# Highest price anywhere in the NEM in each interval (stress spreads between states)
df["nem_max_price"] = df.groupby("interval_start_at")["price_aud_per_mwh"].transform("max")

HORIZON = 6  # predict 30 minutes ahead: features only use data from 6 or more intervals earlier
g = df.groupby("region_code")
df["price_30m_ago"] = g["price_aud_per_mwh"].shift(HORIZON)
df["max_price_prev_1h"] = g["price_aud_per_mwh"].transform(lambda s: s.shift(HORIZON).rolling(12, min_periods=1).max())
df["max_price_prev_24h"] = g["price_aud_per_mwh"].transform(lambda s: s.shift(HORIZON).rolling(288, min_periods=1).max())
df["nem_max_price_30m_ago"] = g["nem_max_price"].shift(HORIZON)
df["demand_30m_ago"] = g["demand_mw"].shift(HORIZON)
df["demand_change_1h"] = df["demand_30m_ago"] - g["demand_mw"].shift(HORIZON + 12)

# The first 18 intervals of each state have no history yet, so drop them (90 rows)
model_df = df.dropna(subset=["price_30m_ago", "demand_change_1h"]).reset_index(drop=True)

feature_cols = ["hour_of_day", "day_of_week", "is_weekend",
                "price_30m_ago", "max_price_prev_1h", "max_price_prev_24h",
                "nem_max_price_30m_ago", "demand_30m_ago", "demand_change_1h"]
features = pd.get_dummies(model_df[feature_cols + ["region_code"]], columns=["region_code"], dtype=int)
target = model_df["is_spike"]

cutoff = pd.Timestamp("2026-08-01")
is_train = model_df["interval_start_at"] < cutoff
X_train, X_test = features[is_train], features[~is_train]
y_train, y_test = target[is_train], target[~is_train]

print(f"Train rows: {len(X_train):,}  Test rows: {len(X_test):,}")
print(f"Train spike rate: {y_train.mean():.2%}  Test spike rate: {y_test.mean():.2%}  Test spikes: {y_test.sum()}")

In [0]:
# train and compare the models

import mlflow
import lightgbm as lgb
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, average_precision_score
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

mlflow.autolog(silent=True)
spike_ratio = (y_train == 0).sum() / (y_train == 1).sum()

# July (the last training month) is held out for early stopping and for choosing the cut-off in cell 5
val_start = pd.Timestamp("2026-07-01")
train_dates = model_df.loc[is_train, "interval_start_at"]
fit_mask = train_dates < val_start
X_fit, y_fit = X_train[fit_mask], y_train[fit_mask]
X_val, y_val = X_train[~fit_mask], y_train[~fit_mask]

candidates = {
    "baseline_no_spike": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "logreg_smote": make_imb_pipeline(StandardScaler(), SMOTE(sampling_strategy=0.1, random_state=42), LogisticRegression(max_iter=1000)),
    "random_forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=20, class_weight="balanced", n_jobs=-1, random_state=42),
    "extra_trees": ExtraTreesClassifier(n_estimators=200, min_samples_leaf=20, class_weight="balanced", n_jobs=-1, random_state=42),
    "gradient_boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
    "xgboost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, scale_pos_weight=spike_ratio, eval_metric="aucpr", n_jobs=-1, random_state=42),
    "lightgbm": LGBMClassifier(n_estimators=300, learning_rate=0.05, scale_pos_weight=spike_ratio, verbose=-1, random_state=42),
    "gradient_boosting_tuned": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                                                              min_samples_leaf=200, l2_regularization=1.0, random_state=42),
    "lightgbm_tuned": LGBMClassifier(n_estimators=2000, learning_rate=0.02, num_leaves=15, min_child_samples=200,
                                     subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                                     verbose=-1, random_state=42),
}

rows, fitted = [], {}
for name, model in candidates.items():
    with mlflow.start_run(run_name=name):
        if name == "lightgbm_tuned":
            model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], eval_metric="average_precision",
                      callbacks=[lgb.early_stopping(100, verbose=False)])
            mlflow.log_param("best_iteration", model.best_iteration_)
        else:
            model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)
        metrics = {
            "test_precision": precision_score(y_test, pred, zero_division=0),
            "test_recall": recall_score(y_test, pred),
            "test_roc_auc": roc_auc_score(y_test, proba),
            "test_pr_auc": average_precision_score(y_test, proba),
        }
        mlflow.log_metrics(metrics)
        rows.append({"model": name, **metrics})
        fitted[name] = model

# The rule to beat: "it will spike if the price 30 minutes ago was over $300"
rule_pred = (X_test["price_30m_ago"] > 300).astype(int)
rows.append({
    "model": "naive_rule_price_30m_ago",
    "test_precision": precision_score(y_test, rule_pred, zero_division=0),
    "test_recall": recall_score(y_test, rule_pred),
    "test_roc_auc": roc_auc_score(y_test, X_test["price_30m_ago"]),
    "test_pr_auc": average_precision_score(y_test, X_test["price_30m_ago"]),
})

results_df = pd.DataFrame(rows).sort_values("test_pr_auc", ascending=False).reset_index(drop=True)
display(results_df)

best_name = results_df.loc[results_df["model"].isin(list(fitted)), "model"].iloc[0]
best_model = fitted[best_name]
print("Best model:", best_name)

In [0]:
# choose the warning cut-off on July, then check it on the test months 

import numpy as np
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve

# Refit a copy of the best model on November to June only, so July is unseen when picking the cut-off
if best_name == "lightgbm_tuned":
    val_model = clone(best_model).set_params(n_estimators=best_model.best_iteration_)
else:
    val_model = clone(best_model)
val_model.fit(X_fit, y_fit)
val_proba = val_model.predict_proba(X_val)[:, 1]

prec, rec, thresholds = precision_recall_curve(y_val, val_proba)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.argmax(f1[:-1])
threshold = float(thresholds[best_idx])
print(f"Cut-off chosen on July: {threshold:.3f}  (July precision {prec[best_idx]:.2f}, recall {rec[best_idx]:.2f})")

# Apply that cut-off to the test months, and compare with the rule
test_proba = best_model.predict_proba(X_test)[:, 1]
test_warn = (test_proba >= threshold).astype(int)
print(f"{best_name} on test:  precision {precision_score(y_test, test_warn, zero_division=0):.2f}  "
      f"recall {recall_score(y_test, test_warn):.2f}  warnings {test_warn.sum():,}")
print(f"Naive rule on test:  precision {precision_score(y_test, rule_pred, zero_division=0):.2f}  "
      f"recall {recall_score(y_test, rule_pred):.2f}  warnings {rule_pred.sum():,}")

with mlflow.start_run(run_name=f"{best_name}_threshold"):
    mlflow.log_param("model", best_name)
    mlflow.log_metric("threshold", threshold)
    mlflow.log_metric("test_precision_at_threshold", precision_score(y_test, test_warn, zero_division=0))
    mlflow.log_metric("test_recall_at_threshold", recall_score(y_test, test_warn))

In [0]:
# Important features, saved as gold table 

from sklearn.inspection import permutation_importance

imp = permutation_importance(best_model, X_test, y_test, scoring="average_precision",
                             n_repeats=3, random_state=42, n_jobs=-1)
importance_df = (pd.DataFrame({"feature": X_test.columns,
                               "importance": imp.importances_mean,
                               "importance_std": imp.importances_std})
                 .sort_values("importance", ascending=False)
                 .reset_index(drop=True))
importance_df["model_name"] = best_name
display(importance_df)

(spark.createDataFrame(importance_df)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("workspace.gold.spike_feature_importance"))
print("Saved workspace.gold.spike_feature_importance")

In [0]:
# score every interval and save the predictions for Power BI

all_proba = best_model.predict_proba(features)[:, 1]

pred_df = pd.DataFrame({
    "region_code": model_df["region_code"],
    "interval_start_at": model_df["interval_start_at"],
    "price_aud_per_mwh": model_df["price_aud_per_mwh"],
    "is_spike": model_df["is_spike"].astype(int),
    "spike_probability": all_proba.astype(float),
    "spike_warning": (all_proba >= threshold).astype(int),
    "rule_warning": (model_df["price_30m_ago"] > 300).astype(int),
    "dataset": np.where(is_train, "train", "test"),
    "model_name": best_name,
})

(spark.createDataFrame(pred_df)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("workspace.gold.price_spike_predictions"))
print(f"Saved {len(pred_df):,} rows to workspace.gold.price_spike_predictions")